### **Fraud Detection in Financial Transactions — Exploratory Data Analysis**
O objetivo desta análise é explorar um conjunto de dados de transações financeiras para identificar padrões associados a atividades fraudulentas.

**Nesta etapa inicial realizamos uma análise exploratória para entender:**
- distribuição das transações
- taxa de fraude
- padrões por categoria de comerciante
- comportamento do valor das transações
- padrões temporais de fraude

Esses insights ajudam a orientar as etapas seguintes de engenharia de features e construção de modelos de detecção de fraude.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

caminho = "/Workspace/Users/henriquegmour4@gmail.com/Fraud Detection in Financial Transactions using PySpark/fraud-detection-spark/data/raw/credit_card_fraud.csv"

fraud_df = spark.read.csv(caminho, header=True, inferSchema=True)

fraud_df.createOrReplaceTempView("fraud_transactions")

In [0]:
# Total de linhas e colunas
fraud_df.count()
fraud_df.printSchema()

**1. 
Data Overview**

In [0]:
# Entendendo o dataset
spark.sql("""
SELECT COUNT(*) AS total_transactions
FROM fraud_transactions
""").show()
print(f"Dimensões do dataset: {fraud_df.count()} linhas x {len(fraud_df.columns)} colunas")
print(f"\nTipos de dados:")
fraud_df.dtypes

**2. Fraud distribution**

In [0]:
# Transacoes que possuem um valor médio maior que as não fraudes
spark.sql("""
SELECT is_fraud, COUNT(*) AS total_transactions
FROM fraud_transactions
GROUP BY is_fraud          
""").show()

# Diferença entre as transações fraudes e não fraudes
spark.sql("""
SELECT
COUNT(*) AS total_transactions,
SUM(is_fraud) AS total_fraud,
ROUND((SUM(is_fraud) / COUNT(*)) * 100, 4) AS fraud_rate_percent
FROM fraud_transactions
""").show()
total = fraud_df.count()
frauds = fraud_df.filter(col("is_fraud") == 1).count()
print(f"\nDataset desbalanceado: {frauds/total*100:.2f}% fraudes")
print("Será necessário balanceamento na modelagem")

**3.Análise do valor das transações**

In [0]:
# Quais as categorias de transações que mais possuem fraudes
spark.sql("""
SELECT is_fraud, COUNT(*) AS total_transactions,
AVG(amount) as avg_transaction_value,
min(amount) as min_transaction_value,
max(amount) as max_transaction_value
FROM fraud_transactions
GROUP BY is_fraud          
""").show()

spark.sql("""
    SELECT 
        is_fraud,
        COUNT(*) as transactions,
        MIN(amount) as min_amount,
        PERCENTILE_APPROX(amount, 0.25) as q1,
        PERCENTILE_APPROX(amount, 0.5) as median,
        PERCENTILE_APPROX(amount, 0.75) as q3,
        MAX(amount) as max_amount,
        ROUND(AVG(amount), 2) as avg_amount,
        ROUND(STDDEV(amount), 2) as std_amount
    FROM fraud_transactions
    GROUP BY is_fraud
""").show()

**4. Fraude por categoria de comerciante**

In [0]:
# Categoria com mais fraudes e taxa de fraude
spark.sql("""
    SELECT 
        merchant_category,
        COUNT(*) as total_transactions,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) as fraud_count,
        ROUND(AVG(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) * 100, 2) as fraud_rate_percent,
        ROUND(AVG(amount), 2) as avg_amount
    FROM fraud_transactions
    GROUP BY merchant_category
    ORDER BY fraud_rate_percent DESC
""").show()

In [0]:
# Transacao por hora e taxa de fraude
spark.sql("""
    SELECT transaction_hour,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_cases,
        ROUND(SUM(is_fraud)/COUNT(*)*100,2) AS fraud_rate
    FROM fraud_transactions
    GROUP BY transaction_hour
    ORDER BY fraud_rate DESC
""").show()

In [0]:
# Transacao por hora e taxa de fraude
spark.sql("""
    SELECT foreign_transaction,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS total_transactions,
        ROUND(SUM(is_fraud)/COUNT(*)*100,2) AS fraud_rate 
    FROM fraud_transactions
    GROUP BY foreign_transaction         
""").show()

In [0]:
# Categorias com mais fraudes
spark.sql("""
SELECT merchant_category,
SUM(is_fraud) AS total_fraud
FROM fraud_transactions
GROUP BY merchant_category
ORDER BY total_fraud DESC 
LIMIT 10 
""").show()